# HCDE notebook C: base model and v1 on the public-benchmark test set

The held-out public tasks (MBPP, HumanEval, CodeAlpaca-20k, DS-1000, FinQA, ConvFinQA) for the base model with the one-line tag, the base model with the full rubric, and model v1; normally and under pressure. Notebook A does the same for v2. About 1.5 hours.

**Run it after `prepare_v2_data.sh` has finished and been pushed** (it needs `public_test.jsonl`).

**Settings:** **GPU T4 x2** · Internet **on** · **Add Input →** the same v1 adapter dataset as notebook B.

**Add-ons → Secrets** (add each and **tick it** for this notebook):

| Secret | Value |
| --- | --- |
| `TEACHER_BASE_URL` | `https://api.groq.com/openai/v1` |
| `TEACHER_API_KEY` | your Groq key(s), comma-separated |
| `TEACHER_MODEL` | `openai/gpt-oss-120b` |
| `JUDGE_MODEL` | `qwen/qwen3.8-27b` (same as in your `.env`) |
| `JUDGE_EXTRA` | `{"reasoning_effort": "none"}` |
| `JUDGE_MAX_TOKENS` | `600` |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |

**How to run:** **Save Version → Save & Run All (Commit)**, then download `hcde_public/hcde-public-results.zip` from the version's Output.

In [ ]:
import os, subprocess, time, glob, json, shutil
WORK = "/kaggle/working"
REPO = f"{WORK}/AnchorAI"
BRANCH = "dev/dulanja"
OUTDIR = f"{WORK}/hcde_public"          # everything worth keeping ends up here
LOGS = f"{OUTDIR}/logs"
os.makedirs(LOGS, exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
def secret(name):
    try:
        return _secrets.get_secret(name)
    except Exception:
        return ""

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

loaded = [k for k in ["TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
                      "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS", "HF_TOKEN"] if secret(k)]
for k in loaded:
    os.environ[k] = secret(k)
HAVE_API = "TEACHER_BASE_URL" in loaded and "TEACHER_API_KEY" in loaded
print("secrets loaded:", loaded or "none")
if not HAVE_API:
    print("WARNING: no Groq secrets -> no judging (replies are still saved)")

subprocess.run("pip install -q -U peft bitsandbytes accelerate && pip uninstall -q -y torchao", shell=True)
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
print("GPUs:", GPUS, "(both are used in parallel)" if GPUS > 1 else "(steps run one after another)")

PY = "python -m components.c4_hcde.training"
DATA = "components/c4_hcde/training/data/dataset"

def start(name, cmds, gpu=None):
    """Run shell commands one after another in the background (a failed command does not stop the
    next one). Output goes to logs/<name>.log."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = open(f"{LOGS}/{name}.log", "w")
    p = subprocess.Popen(" ; ".join(cmds), shell=True, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
    p.name = name
    print(f"started {name}" + (f" on GPU {gpu}" if gpu is not None else ""))
    return p

def tail(name, n=2):
    try:
        lines = [l.rstrip() for l in open(f"{LOGS}/{name}.log", errors="replace").read().replace("\r", "\n").splitlines() if l.strip()]
        return " | ".join(lines[-n:])[-300:]
    except FileNotFoundError:
        return ""

def wait(procs, every=300):
    """Block until all background jobs finish, printing progress every few minutes."""
    t0 = last = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        if time.time() - last < every and any(p.poll() is None for p in procs):
            continue
        last = time.time()
        print(f"--- {int((time.time() - t0) / 60)} min")
        for p in procs:
            print(f"  {p.name}: {'running' if p.poll() is None else 'done'} :: {tail(p.name)}", flush=True)
    for p in procs:
        print(f"{p.name}: exit code {p.returncode}  :: {tail(p.name, 3)}")

def run_on_gpus(chains):
    """chains = list of (name, [commands]); one chain per GPU, in parallel when there are 2 GPUs."""
    if GPUS > 1:
        wait([start(n, c, gpu=i % GPUS) for i, (n, c) in enumerate(chains)])
    else:
        for n, c in chains:
            wait([start(n, c, gpu=0)])

In [ ]:
found = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not found:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        subprocess.run(["unzip", "-q", "-o", z, "-d", f"{WORK}/v1"])
    found = glob.glob(f"{WORK}/v1/**/adapter_config.json", recursive=True)
V1 = os.path.dirname(found[0]) if found else None
print("v1 adapter:", V1 or "NOT FOUND (add the dataset); only base-model conditions will run")
print("public test set:", os.path.exists(f"{DATA}/public_test.jsonl"))
ALL = f"--adapter {V1} --tuned-name tuned_v1" if V1 else "--conditions base_tag base_rubric"
PB = f"--test {DATA}/public_test.jsonl --elements {DATA}/public_elements.json"
run_on_gpus([
    ("eval_public", [f"{PY}.evaluate {ALL} {PB} --out {OUTDIR}/eval-public"]),
    ("eval_public_pressure", [f"{PY}.evaluate {ALL} {PB} --pressure --out {OUTDIR}/eval-public-pressure"]),
])

In [ ]:
if HAVE_API:
    jobs = []
    for d, elements in [(f"{OUTDIR}/eval-public", f"{DATA}/public_elements.json"),
                     (f"{OUTDIR}/eval-public-pressure", f"{DATA}/public_elements.json")]:
        if os.path.exists(f"{d}/generations.jsonl"):
            name = "judge_" + d.rstrip("/").split("/")[-1]
            # run twice: the second pass only re-scores rows the first one could not (rate limits)
            cmd = f"{PY}.evaluate --from-generations {d}/generations.jsonl --elements {elements} --out {d} --judge"
            jobs.append(start(name, [cmd, cmd]))
    wait(jobs)
else:
    print("no Groq secrets: replies are saved, judge them later on the Mac with evaluate --from-generations ... --judge")

subprocess.run(f"{PY}.stats --set public {OUTDIR}/eval-public/generations.jsonl {OUTDIR}/eval-public-pressure/generations.jsonl "
               f"--out {OUTDIR}/stats.md", shell=True)
print(open(f"{OUTDIR}/stats.md").read() if os.path.exists(f"{OUTDIR}/stats.md") else "no stats")
os.chdir(OUTDIR)
subprocess.run("zip -qr hcde-public-results.zip eval-public eval-public-pressure stats.md logs", shell=True)
os.chdir(REPO)
for f in glob.glob(f"{OUTDIR}/*.zip"):
    print(f"{os.path.getsize(f) / 1e6:.1f} MB  {f}")